# About:

Script used to validate trained CAM policy:
    
    + Plot action distribution 
    + Extract SHAP values
    



In [ ]:

# Standard Library Imports
import os
import sys
import pickle
import shutil

# Third Party Library Imports
import numpy as np
import torch
import pandas as pd
import torch as T
import shap
import matplotlib.pyplot as plt
import dill
from IPython.display import display, Javascript


# Local Imports 
from leo_gym.gyms.cam_gym import CamEnv, CamEnvConfig
from leo_gym.orbit.dynamics.dynamics import DynamicsConfig
from leo_gym.satellite.sat_debris_cluster import SatDebrisClusterConfig
from leo_gym.utils.utils import seed_all, create_dir
from leo_gym.rl_algorithms.h_ppo.h_ppo_agent import Agent
from leo_gym.rl_algorithms.h_ppo.config import PPOConfig



In [ ]:
# Network and environment paths

env_cfg = "./trained_policy_net_cfg/env_cfg.json"
policy_file_path = "./trained_policy_net_cfg/policynet.pth"
critic_file_path = "./trained_policy_net_cfg/valuenet.pth"
ppo_config = "./trained_policy_net_cfg/ppo_cfg.json"


In [ ]:
# Dataset path 
mc_csv_path = "./monte_carlo_data/background_data_states.csv"


In [ ]:
import json, numpy as np, re


def make_env():
    seed = np.random.randint(0, 2**32)
    return CamEnv(env_cfg, seed)

num_envs = 1
env = make_env()
obs = env.reset()


In [ ]:
# Load Agent 
device = 'cpu'


from leo_gym.rl_algorithms.h_ppo.h_ppo_agent import Agent


ppo = Agent(
    env_obs=env.observation_space,
    env_actions=env.action_space,
    ppo_cfg=ppo_config,
    train=False,
    policy_file_path=policy_file_path,
    critic_file_path=critic_file_path,
    device=device
)
model = ppo.policy_net


In [ ]:

background_data = np.loadtxt(mc_csv_path, delimiter=",")
print(background_data)


In [ ]:
feature_names = [r"$u_{\mathrm{p}}$", r"$i_{\mathrm{p}}$",  r"$\Omega_{\mathrm{p}}$", "$a\delta\lambda$", "$a\delta e_x$", "$a\delta e_y$", r"$u_{\mathrm{s}}$", r"$i_{\mathrm{s}}$", r"$\Omega_{\mathrm{s}}$", r"$\Delta t_{\mathrm{TCA}}$","$\det\mathbf{C}$", "$l$", r"$P^{\mathrm{max}}_{\mathrm{c}}$ ", r"$\delta r_\xi$", r"$\delta r_\zeta$"]
# feature_names = [r"$u_{\text{p}}$", "$a\delta\lambda$", "$a\delta e_x$", "$a\delta e_y$",r"$\Delta t_{\text{TCA}}$", r"$P^{\text{max}}_{\text{c}}$ "]


output_names = [ "0", "$+f_r$","$-f_r$", r"$\Delta t_\mathrm{del}$", r"$\Delta t_{\mathrm{dur}}$"]
len(feature_names)

In [ ]:
from torch import nn
import torch 

class SHAPWrapper(nn.Module):
    def __init__(self, policy_net, min_val, max_val):
        super().__init__()
        self.policy_net = policy_net
        self.register_buffer("min_val", torch.tensor(min_val, dtype=torch.float32))
        self.register_buffer("max_val", torch.tensor(max_val, dtype=torch.float32))

    def forward(self, states):
        dist_dis, dist_cont = self.policy_net(states)
        probs = (dist_dis.probs * 100.0).view(states.size(0), -1)
        cont = dist_cont.mean
        act_scale = (self.max_val - self.min_val) / 2.0
        act_bias = (self.max_val + self.min_val) / 2.0
        cont = (cont.clamp(-1, 1) * act_scale + act_bias).view(states.size(0), -1)
        return torch.cat([probs, cont], dim=1)


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
min_val = [9, 1]
max_val = [55, 41]

policy_net = model
policy_net.eval()

wrapper = SHAPWrapper(policy_net, min_val, max_val).eval().to(device)


In [ ]:
import numpy as np
import torch
import shap

K = 100
n_test = 1000

background_subset = np.asarray(shap.sample(background_data, K), dtype=np.float32)
idx = np.random.choice(background_data.shape[0], n_test, replace=False)
x_test_np = np.asarray(background_data[idx], dtype=np.float32)

x_bg = torch.tensor(background_subset, dtype=torch.float32, device=device)
x_test = torch.tensor(x_test_np, dtype=torch.float32, device=device)

wrapper = wrapper.eval().to(device)

explainer = shap.DeepExplainer(wrapper, x_bg)
shap_values = explainer.shap_values(x_test, check_additivity=False)

shap_exp = shap.Explanation(
    values=shap_values,
    data=x_test.detach().cpu().numpy(),
    feature_names=feature_names
)


In [ ]:
background_data.shape

In [ ]:
os.makedirs("temp", exist_ok=True)
os.chdir("temp")


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pickle

bg_tensor = torch.tensor(background_data, dtype=torch.float32)

with torch.no_grad():
    outputs = wrapper(bg_tensor)

outputs = outputs.detach().cpu().numpy().reshape(outputs.shape[0], -1)
outputs
cont = outputs[:, 3:]  

plt.figure(figsize=(8, 4))
bins = 30

for i in range(cont.shape[1]):
    counts, edges = np.histogram(cont[:, i], bins=bins, density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    plt.plot(centers, counts, label=output_names[i+3])
    plt.fill_between(centers, counts, alpha=0.3)

plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Density Curve of Firing Delay & Duration')
plt.legend()
plt.tight_layout()
plt.savefig("cont_act_distr.png",dpi=300) 


In [ ]:
outputs

In [ ]:


shap_values_all = shap.Explanation(values=shap_values, 
                                    data=x_test_np,
                                    feature_names=feature_names,
                                    output_names=output_names)


In [ ]:
shap_values_all.shape

In [ ]:
from leo_gym.utils.matplot_style_cfg import *

save_file_name2 = [
    "shap_f_0_beeswarm_plot",
    "shap_f_beeswarm_plot",
    "shap_f_neg_beeswarm_plot",
    "shap_dt0_beeswarm_plot",
    "shap_dtman_beeswarm_plot"
]

for i in range(len(save_file_name2)):
    print(output_names[i])
        
    shap.plots.beeswarm(shap_values_all[:,:,output_names[i]], show=False)
    fig, ax = plt.gcf(), plt.gca()
    ax.tick_params(axis='both', colors='black')  
    ax.xaxis.label.set_color('black')            
    ax.yaxis.label.set_color('black')            
    cb_ax = fig.axes[1] 

    fig = plt.gcf()
    fig.set_size_inches(5, 5)


    plt.savefig(f"{save_file_name2[i]}.png", bbox_inches="tight", format="png")
    # plt.savefig(f"{save_file_name2[i]}.pgf", bbox_inches="tight", format="pgf")
    print(save_file_name2[i])
    plt.close(fig)
